# ЛР № 3. Профилирование и качество данных

**Проект:** анализ рынка краткосрочной аренды в Барселоне (Inside Airbnb)  
**Команда:** Козарь Дмитрий, Кибкало Никита, Шатило Владислав, ПИН-241  
**Датасет:** listings.csv.gz, один ряд — одно объявление.

Ноутбук читает исходный объект непосредственно из `raw` в SeaweedFS через DuckDB. Сначала задайте фактический S3-путь из успешного запуска DAG ЛР № 2 и выполните все ячейки сверху вниз. Не меняйте и не перезаписывайте исходный объект `raw`.


## 1. Параметры и подключение

In [ ]:
import os
import re
from pathlib import PurePosixPath
import duckdb
import pandas as pd
from IPython.display import display

# Вставьте полный путь из успешного запуска DAG ЛР № 2.
RAW_S3_PATH = os.getenv(
    "RAW_S3_PATH",
    "s3://raw/inside_airbnb_barcelona/ingested_on=2026-10-30/listings.csv.gz"
)
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "localhost:8333")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY", "s3admin")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY", "s3admin123")
S3_USE_SSL = os.getenv("S3_USE_SSL", "false").lower() == "true"

if "YYYY-MM-DD" in RAW_S3_PATH:
    raise ValueError("Укажите фактический RAW_S3_PATH из Airflow; не оставляйте YYYY-MM-DD.")

m = re.search(r"ingested_on=(\d{4}-\d{2}-\d{2})", RAW_S3_PATH)
if not m:
    raise ValueError("В RAW_S3_PATH должен присутствовать сегмент ingested_on=YYYY-MM-DD.")
INGESTED_ON = m.group(1)
DATASET_SLUG = "inside-airbnb-barcelona"
OUTPUT_PREFIX = f"s3://staging/{DATASET_SLUG}/source_ingested_on={INGESTED_ON}"

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
endpoint = S3_ENDPOINT.removeprefix("http://").removeprefix("https://").rstrip("/")
con.execute(f"SET s3_endpoint='{endpoint}'")
con.execute(f"SET s3_access_key_id='{S3_ACCESS_KEY}'")
con.execute(f"SET s3_secret_access_key='{S3_SECRET_KEY}'")
con.execute("SET s3_url_style='path'")
con.execute(f"SET s3_use_ssl={'true' if S3_USE_SSL else 'false'}")
print("Raw:", RAW_S3_PATH)
print("Output:", OUTPUT_PREFIX)


## 2. Чтение исходных данных и первичный профиль

In [ ]:
# listings.csv.gz читается напрямую из S3. Исходный объект не изменяется.
con.execute(f"CREATE OR REPLACE VIEW raw_listings AS SELECT * FROM read_csv_auto('{RAW_S3_PATH}', header=true, sample_size=-1, ignore_errors=false)")
display(con.execute("SELECT * FROM raw_listings LIMIT 5").df())
schema = con.execute("DESCRIBE raw_listings").df()
display(schema)
row_count = con.execute("SELECT COUNT(*) AS rows FROM raw_listings").fetchone()[0]
column_count = len(schema)
print(f"Строк: {row_count:,}; полей: {column_count}")


In [ ]:
# Поля проекта и пропуски. Пропуск neighbourhood здесь может означать отсутствие
# классификации района, а не отсутствие самого объявления.
required = {"id","price","neighbourhood","room_type","latitude","longitude",
            "number_of_reviews","availability_365","host_id",
            "calculated_host_listings_count"}
missing_columns = required - set(schema["column_name"])
if missing_columns:
    raise ValueError(f"В исходном файле отсутствуют ожидаемые поля: {sorted(missing_columns)}")

profile = con.execute("""
SELECT
  COUNT(*) AS checked_rows,
  COUNT(*) FILTER (WHERE id IS NULL) AS missing_id,
  COUNT(*) FILTER (WHERE price IS NULL) AS missing_price,
  COUNT(*) FILTER (WHERE neighbourhood IS NULL OR trim(CAST(neighbourhood AS VARCHAR))='') AS missing_neighbourhood,
  COUNT(*) FILTER (WHERE room_type IS NULL OR trim(CAST(room_type AS VARCHAR))='') AS missing_room_type,
  COUNT(*) FILTER (WHERE latitude IS NULL OR longitude IS NULL) AS missing_coordinates,
  COUNT(*) FILTER (WHERE number_of_reviews IS NULL) AS missing_reviews
FROM raw_listings
""").df()
display(profile)
duplicates = con.execute("""
SELECT COUNT(*) AS rows_in_duplicate_ids
FROM raw_listings
WHERE id IN (SELECT id FROM raw_listings GROUP BY id HAVING COUNT(*) > 1)
""").df()
display(duplicates)
display(con.execute("SELECT room_type, COUNT(*) AS listings FROM raw_listings GROUP BY room_type ORDER BY listings DESC").df())
display(con.execute("SELECT MIN(last_review) AS first_review, MAX(last_review) AS last_review FROM raw_listings").df())


## 3. Правила качества — отдельные проверки

Порог каждого правила — 0 нарушений. Превышение порога фиксируется в сводке; действие к отдельной записи применяется независимо от превышения порога.

In [ ]:
rule_results = []


### `unique_listing_id`
- Поле: `id`; проверяются все строки raw.
- Условие нарушения: `id IS NULL OR _id_rn > 1`. Порог: **0 нарушений**. Критичность: `critical`; действие: `quarantine`.
- Обоснование: Дубли искажают количество предложений и доли рынка.

In [ ]:
rule_id = 'unique_listing_id'
condition = 'id IS NULL OR _id_rn > 1'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'critical', 'action': 'quarantine'})
display(pd.DataFrame([rule_results[-1]]))


### `price_nonnegative`
- Поле: `price`; проверяются все строки raw.
- Условие нарушения: `_price IS NULL OR _price < 0`. Порог: **0 нарушений**. Критичность: `critical`; действие: `quarantine`.
- Обоснование: Некорректная цена искажает сравнение стоимости по районам.

In [ ]:
rule_id = 'price_nonnegative'
condition = '_price IS NULL OR _price < 0'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'critical', 'action': 'quarantine'})
display(pd.DataFrame([rule_results[-1]]))


### `neighbourhood_present`
- Поле: `neighbourhood`; проверяются все строки raw.
- Условие нарушения: `_missing_neighbourhood`. Порог: **0 нарушений**. Критичность: `warning`; действие: `flag`.
- Обоснование: Пропуск района ограничивает районные срезы.

In [ ]:
rule_id = 'neighbourhood_present'
condition = '_missing_neighbourhood'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'warning', 'action': 'flag'})
display(pd.DataFrame([rule_results[-1]]))


### `room_type_allowed`
- Поле: `room_type`; проверяются все строки raw.
- Условие нарушения: `_bad_room_type`. Порог: **0 нарушений**. Критичность: `critical`; действие: `quarantine`.
- Обоснование: Неизвестная категория искажает распределение типов жилья.

In [ ]:
rule_id = 'room_type_allowed'
condition = '_bad_room_type'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'critical', 'action': 'quarantine'})
display(pd.DataFrame([rule_results[-1]]))


### `coordinates_in_range`
- Поле: `latitude, longitude`; проверяются все строки raw.
- Условие нарушения: `_bad_coordinates`. Порог: **0 нарушений**. Критичность: `warning`; действие: `flag`.
- Обоснование: Некорректные координаты мешают пространственному анализу.

In [ ]:
rule_id = 'coordinates_in_range'
condition = '_bad_coordinates'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'warning', 'action': 'flag'})
display(pd.DataFrame([rule_results[-1]]))


### `reviews_nonnegative`
- Поле: `number_of_reviews`; проверяются все строки raw.
- Условие нарушения: `_reviews IS NULL OR _reviews < 0`. Порог: **0 нарушений**. Критичность: `warning`; действие: `flag`.
- Обоснование: Отзывы используются как приближённый показатель спроса.

In [ ]:
rule_id = 'reviews_nonnegative'
condition = '_reviews IS NULL OR _reviews < 0'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'warning', 'action': 'flag'})
display(pd.DataFrame([rule_results[-1]]))


### `availability_in_range`
- Поле: `availability_365`; проверяются все строки raw.
- Условие нарушения: `_availability IS NULL OR _availability NOT BETWEEN 0 AND 365`. Порог: **0 нарушений**. Критичность: `warning`; действие: `flag`.
- Обоснование: Доступность должна быть числом дней в диапазоне 0–365.

In [ ]:
rule_id = 'availability_in_range'
condition = '_availability IS NULL OR _availability NOT BETWEEN 0 AND 365'
checked_rows, violations = con.execute(f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condition}) FROM profiled").fetchone()
threshold = 0
rule_results.append({'id': rule_id, 'checked_rows': checked_rows, 'violations': violations, 'violation_share': violations / checked_rows if checked_rows else 0.0, 'threshold': threshold, 'threshold_exceeded': violations > threshold, 'severity': 'warning', 'action': 'flag'})
display(pd.DataFrame([rule_results[-1]]))


In [ ]:
rules_df = pd.DataFrame(rule_results)
display(rules_df)


## 4. Формирование staging и quarantine

In [ ]:
# Карантин — объединение всех критических причин, каждая запись только один раз.
# quality_rule_ids содержит список идентификаторов через запятую.
critical_conditions = {
    "unique_listing_id": "(_id IS NULL OR _id_rn > 1)",
    "price_nonnegative": "(_price IS NULL OR _price < 0)",
    "room_type_allowed": "(_bad_room_type)"
}
reason_parts = [f"CASE WHEN {cond} THEN '{rid}' END" for rid, cond in critical_conditions.items()]
ids_expr = " || ',' || ".join([f"COALESCE({p}, '')" for p in reason_parts])
# Удаляем пустые элементы при сборке причин; исходные столбцы сохраняются.
con.execute(f"""
CREATE OR REPLACE TABLE quarantine_result AS
SELECT * EXCLUDE (_id,_price,_latitude,_longitude,_reviews,_availability,_id_rn,
                  _missing_neighbourhood,_bad_room_type,_bad_coordinates),
       array_to_string(list_filter([{', '.join(reason_parts)}], x -> x IS NOT NULL), ',') AS quality_rule_ids,
       array_to_string(list_filter([
          CASE WHEN _id IS NULL OR _id_rn > 1 THEN 'Отсутствует или повторяется id объявления' END,
          CASE WHEN _price IS NULL OR _price < 0 THEN 'Цена отсутствует, нечисловая или отрицательная' END,
          CASE WHEN _bad_room_type THEN 'Тип жилья отсутствует или не входит в допустимый справочник' END
       ], x -> x IS NOT NULL), '; ') AS quarantine_reason
FROM profiled
WHERE {' OR '.join(critical_conditions.values())}
""")
con.execute("""
CREATE OR REPLACE TABLE staging_result AS
SELECT * EXCLUDE (_id,_price,_latitude,_longitude,_reviews,_availability,_id_rn,
                  _missing_neighbourhood,_bad_room_type,_bad_coordinates),
       _price AS price_normalized,
       _latitude AS latitude_normalized,
       _longitude AS longitude_normalized,
       _missing_neighbourhood AS quality_flag_missing_neighbourhood,
       _bad_coordinates AS quality_flag_bad_coordinates,
       (_reviews IS NULL OR _reviews < 0) AS quality_flag_bad_reviews,
       (_availability IS NULL OR _availability NOT BETWEEN 0 AND 365) AS quality_flag_bad_availability
FROM profiled
WHERE NOT (_id IS NULL OR _id_rn > 1 OR _price IS NULL OR _price < 0 OR _bad_room_type)
""")
raw_n = con.execute("SELECT COUNT(*) FROM raw_listings").fetchone()[0]
staging_n = con.execute("SELECT COUNT(*) FROM staging_result").fetchone()[0]
quarantine_n = con.execute("SELECT COUNT(*) FROM quarantine_result").fetchone()[0]
print({"raw":raw_n,"staging":staging_n,"quarantine":quarantine_n,"balance_ok":raw_n == staging_n + quarantine_n})
display(con.execute("SELECT quality_rule_ids, quarantine_reason, COUNT(*) AS rows FROM quarantine_result GROUP BY ALL ORDER BY rows DESC LIMIT 10").df())


In [ ]:
# Запись в SeaweedFS. COPY создаёт Parquet-объекты по заданным путям.
staging_path = f"{OUTPUT_PREFIX}/staging.parquet"
quarantine_path = f"{OUTPUT_PREFIX}/quarantine.parquet"
con.execute(f"COPY staging_result TO '{staging_path}' (FORMAT PARQUET)")
con.execute(f"COPY quarantine_result TO '{quarantine_path}' (FORMAT PARQUET)")
print("Записано:", staging_path, quarantine_path)


## 5. Повторное чтение и итоговые проверки

In [ ]:
# Проверяем оба результата чтением напрямую из S3.
staging_check = con.execute(f"SELECT COUNT(*) FROM read_parquet('{staging_path}')").fetchone()[0]
quarantine_check = con.execute(f"SELECT COUNT(*) FROM read_parquet('{quarantine_path}')").fetchone()[0]
assert staging_check == staging_n
assert quarantine_check == quarantine_n
assert raw_n == staging_n + quarantine_n, "Баланс raw = staging + quarantine не соблюдён"

# В staging не должно быть нарушений критических правил.
critical_violations = con.execute(f"""
SELECT
 COUNT(*) FILTER (WHERE TRY_CAST(price_normalized AS DOUBLE) IS NULL OR TRY_CAST(price_normalized AS DOUBLE) < 0) AS invalid_price,
 COUNT(*) FILTER (WHERE room_type IS NULL OR trim(CAST(room_type AS VARCHAR)) NOT IN
   ('Entire home/apt','Private room','Shared room','Hotel room')) AS invalid_room_type
FROM read_parquet('{staging_path}')
""").df()
display(critical_violations)
assert int(critical_violations.iloc[0]["invalid_price"]) == 0
assert int(critical_violations.iloc[0]["invalid_room_type"]) == 0
print({"raw":raw_n,"staging_first_run":staging_n,"quarantine_first_run":quarantine_n,
       "staging_readback":staging_check,"quarantine_readback":quarantine_check})
